In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

from helpers import init_mpl_fig, save_mpl_fig

In [ ]:
df = pd.read_parquet("gc_human_tweets.parquet").assign(
    mentions=lambda df_: df_["mentions"].apply(
        lambda x: x.tolist() if isinstance(x, np.ndarray) else x
    )
)
df

In [ ]:
# Count mentions per tweet
df["n_mentions"] = df["mentions"].apply(len)

print(
    f"\nTweets with mentions: {(df['n_mentions'] > 0).sum():,} ({(df['n_mentions'] > 0).mean()*100:.1f}%)"
)
print(f"Total mentions: {df['n_mentions'].sum():,}")
print(f"Average mentions per tweet (all): {df['n_mentions'].mean():.2f}")
print(
    f"Average mentions per tweet (with mentions): {df.loc[df['n_mentions'] > 0, 'n_mentions'].mean():.2f}"
)

# Distribution of mentions per tweet
print("\nDistribution of mentions per tweet:")
mention_dist = df["n_mentions"].value_counts().sort_index()
for n_mentions, count in mention_dist.head(10).items():
    pct = (count / len(df)) * 100
    print(f"  {n_mentions} mentions: {count:,} tweets ({pct:.1f}%)")

## Prep network

In [ ]:
# Filter to tweets with mentions
df_with_mentions = df[df["n_mentions"] > 0].copy()
print(f"Building network from {len(df_with_mentions):,} tweets with mentions...")

# Explode mentions to create one row per author-mention pair
print("Exploding mentions...")
df_edges = (
    df_with_mentions[["author_id", "mentions"]]
    .explode("mentions")
    .dropna(subset=["mentions"])
    .assign(
        author=lambda x: x["author_id"].astype(str),
        mentioned=lambda x: x["mentions"].str.lower(),
    )[["author", "mentioned"]]
)

print(f"Total edges (raw): {len(df_edges):,}")

# Count edge frequencies
print("Counting edge frequencies...")
edge_counts = (
    df_edges.groupby(["author", "mentioned"]).size().reset_index(name="weight")
)

print(f"Unique edges: {len(edge_counts):,}")

# Build network from edge list
print("Building NetworkX graph...")
G = nx.from_pandas_edgelist(
    edge_counts,
    source="author",
    target="mentioned",
    edge_attr="weight",
    create_using=nx.DiGraph(),
)

print("\nNetwork constructed:")
print(f"  Nodes: {G.number_of_nodes():,}")
print(f"  Edges: {G.number_of_edges():,}")
print(f"  Density: {nx.density(G):.6f}")

In [ ]:
edge_counts = (
    df_edges.groupby(["author", "mentioned"]).size().reset_index(name="weight")
)

print(f"Unique edges: {len(edge_counts):,}")

# Now continue with the rest...
df_top_mentioned = (
    edge_counts.groupby("mentioned")["weight"]
    .sum()
    .reset_index()
    .rename(columns={"weight": "mentions"})
    .sort_values("mentions", ascending=False)
    .reset_index(drop=True)
)

df_top_mentioned["pct_tweets"] = (df_top_mentioned["mentions"] / len(df)) * 100

print("\nTop 30 most mentioned accounts:")
print(f"{'Rank':<6} {'Account':<30} {'Mentions':>10} {'% of Tweets':>12}")
print("-" * 60)

for rank, row in df_top_mentioned.head(30).iterrows():
    print(
        f"{rank+1:<6} @{row['mentioned']:<29} {row['mentions']:>10,} {row['pct_tweets']:>11.2f}%"
    )

## Plot

In [ ]:
def make_network(
    graph,
    df_top_mentioned,
    n_top_accounts=20,
    figure_size=24,
    spring_k=2,
    spring_iterations=50,
    random_seed=42,
    blue_node_size_mult=15,
    blue_node_alpha=0.5,
    red_node_size_mult=10,
    red_node_alpha=0.32,
    secondary_node_alpha=0.32,
    secondary_node_color="orange",
    edge_alpha=0.06,
    label_font_size=18,
    label_font_color="black",
    secondary_label_color="darkorange",
    use_adjust_text=True,
    adjust_text_expand_points=(1.5, 1.5),
    adjust_text_expand_text=(1.3, 1.3),
    adjust_text_force=(0.5, 0.5),
    secondary_keywords=None,
):
    """
    Create mention network visualization showing top mentioned accounts and their mentioners.

    Parameters
    ----------
    graph : networkx.DiGraph
        Full mention network graph (user_id -> mentioned_account)
    df_top_mentioned : pd.DataFrame
        DataFrame with columns ['mentioned', 'mentions'] sorted by mention count
    n_top_accounts : int, default=20
        Number of top mentioned accounts to visualize
    figure_size : int, default=24
        Size of the square figure in inches
    spring_k : float, default=2
        Optimal distance between nodes in spring layout (higher = more spread)
    spring_iterations : int, default=50
        Number of iterations for layout optimization (higher = better quality)
    random_seed : int, default=42
        Random seed for reproducible layout
    blue_node_size_mult : int, default=15
        Multiplier for blue (user) node sizes
    blue_node_alpha : float, default=0.5
        Transparency for blue nodes (0-1)
    red_node_size_mult : int, default=10
        Multiplier for red (primary) node sizes
    red_node_alpha : float, default=0.32
        Transparency for red (primary) nodes (lower = more glow)
    secondary_node_alpha : float, default=0.32
        Transparency for secondary highlighted nodes
    secondary_node_color : str, default='orange'
        Color for secondary highlighted nodes
    edge_alpha : float, default=0.06
        Transparency for edges (lower = fainter)
    label_font_size : int, default=18
        Font size for account labels
    label_font_color : str, default='black'
        Color for primary node labels
    secondary_label_color : str, default='darkorange'
        Color for secondary node labels
    use_adjust_text : bool, default=True
        Use adjustText library to prevent label overlap
    adjust_text_expand_points : tuple, default=(1.5, 1.5)
        How much to push labels away from nodes
    adjust_text_expand_text : tuple, default=(1.3, 1.3)
        How much to push labels away from each other
    adjust_text_force : tuple, default=(0.5, 0.5)
        Force strength for label movement
    secondary_keywords : list, optional
        Keywords to identify secondary category (e.g., organizations).
        Accounts containing these keywords will be colored with secondary_node_color.
        If None or empty, all top accounts are colored red (primary color only).

    Returns
    -------
    fig : matplotlib.figure.Figure
        The figure object
    ax : matplotlib.axes.Axes
        The axes object
    stats : dict
        Dictionary with visualization statistics

    Examples
    --------
    # All nodes red (no secondary highlighting)
    fig, ax, stats = create_mention_network_viz(G, df_top_mentioned)

    # Highlight organizations in orange
    fig, ax, stats = create_mention_network_viz(
        G, df_top_mentioned,
        secondary_keywords=['youtube', 'gop', 'nyt', 'cnn', 'news']
    )

    # Highlight women in purple
    fig, ax, stats = create_mention_network_viz(
        G, df_top_mentioned,
        secondary_keywords=['hillary', 'alyssa', 'rose', 'oprah'],
        secondary_node_color='purple',
        secondary_label_color='darkviolet'
    )
    """
    # ------------------------------------------------------------
    # ------------------------------------------------------------
    # Pre-plot prep
    # ------------------------------------------------------------
    # ------------------------------------------------------------

    # ============================================================
    # Get top N accounts
    top_accounts = df_top_mentioned.head(n_top_accounts)["mentioned"].tolist()

    # ============================================================
    # Classify primary vs secondary (if secondary_keywords provided)
    if secondary_keywords and len(secondary_keywords) > 0:

        def is_secondary(account):
            account_lower = account.lower()
            return any(keyword in account_lower for keyword in secondary_keywords)

        secondary = [acc for acc in top_accounts if is_secondary(acc)]
        primary = [acc for acc in top_accounts if not is_secondary(acc)]

        print(f"  Primary nodes: {len(primary)}")
        print(f"  Secondary nodes: {len(secondary)}")
    else:
        # No secondary category - all accounts are primary (red)
        primary = top_accounts
        secondary = []
        print(f"  All nodes: {len(primary)} (no secondary highlighting)")

    # ============================================================
    # Create subgraph
    print("  Creating subgraph...")
    ego_nodes = set(top_accounts)
    for account in top_accounts:
        ego_nodes.update([n for n in graph.predecessors(account)])

    ego_graph = graph.subgraph(ego_nodes)
    print(f"  Nodes: {ego_graph.number_of_nodes():,}")
    print(f"  Edges: {ego_graph.number_of_edges():,}")

    # ============================================================
    # Compute layout
    print("  Computing layout...")
    pos = nx.spring_layout(
        ego_graph, k=spring_k, iterations=spring_iterations, seed=random_seed
    )

    # ------------------------------------------------------------
    # ------------------------------------------------------------
    # plot
    # ------------------------------------------------------------
    # ------------------------------------------------------------
    _, ax = init_mpl_fig(aspect_ratio=(figure_size, figure_size))

    # ============================================================
    # Draw user nodes (blue)
    print("  Drawing nodes...")
    user_nodes = [n for n in ego_graph.nodes() if n not in top_accounts]
    user_sizes = [ego_graph.degree(node) * blue_node_size_mult for node in user_nodes]

    nx.draw_networkx_nodes(
        ego_graph,
        pos,
        nodelist=user_nodes,
        node_color="lightblue",
        node_size=user_sizes,
        alpha=blue_node_alpha,
        ax=ax,
    )

    mentioned_nodes = [n for n in ego_graph.nodes() if n in top_accounts]
    mentioned_sizes = [
        ego_graph.degree(node) * (blue_node_size_mult / 60) for node in mentioned_nodes
    ]
    nx.draw_networkx_nodes(
        ego_graph,
        pos,
        nodelist=mentioned_nodes,
        node_color="lightblue",
        node_size=mentioned_sizes,
        alpha=(blue_node_alpha + 0.2),
        ax=ax,
    )

    # ============================================================
    # Draw edges
    print("  Drawing edges...")
    nx.draw_networkx_edges(ego_graph, pos, alpha=edge_alpha, arrows=False, ax=ax)

    # ============================================================
    # Draw mentioned nodes (red)
    primary_sizes = [ego_graph.degree(node) * red_node_size_mult for node in primary]
    nx.draw_networkx_nodes(
        ego_graph,
        pos,
        nodelist=primary,
        node_color="red",
        node_size=primary_sizes,
        alpha=red_node_alpha,
        ax=ax,
    )

    # ============================================================
    # Draw secondary nodes
    if secondary:
        secondary_sizes = [
            ego_graph.degree(node) * red_node_size_mult for node in secondary
        ]
        nx.draw_networkx_nodes(
            ego_graph,
            pos,
            nodelist=secondary,
            node_color=secondary_node_color,
            node_size=secondary_sizes,
            alpha=secondary_node_alpha,
            ax=ax,
        )

    # ============================================================
    # Add labels
    print("  Adding labels...")
    if use_adjust_text:
        try:
            from adjustText import adjust_text

            texts = []
            for node in top_accounts:
                x, y = pos[node]
                # Use secondary color for secondary nodes, primary color otherwise
                label_color = (
                    secondary_label_color if node in secondary else label_font_color
                )
                text = ax.text(
                    x,
                    y,
                    f"@{node}",
                    fontsize=label_font_size,
                    color=label_color,
                    ha="center",
                    va="center",
                    zorder=100,
                )
                texts.append(text)

            adjust_text(
                texts,
                arrowprops=dict(arrowstyle="-", color="gray", lw=0.5, alpha=0.5),
                expand_points=adjust_text_expand_points,
                expand_text=adjust_text_expand_text,
                force_points=adjust_text_force,
                force_text=adjust_text_force,
                ax=ax,
            )
            print("  Labels adjusted")
        except ImportError:
            print("  Warning: adjustText not available, using basic labels")
            use_adjust_text = False

    if not use_adjust_text:
        labels = {
            node: f"@{node}" if node in top_accounts else ""
            for node in ego_graph.nodes()
        }
        nx.draw_networkx_labels(
            ego_graph, pos, labels, font_size=label_font_size, ax=ax
        )

    # ============================================================
    ax.axis("off")
    ax.margins(0)
    plt.tight_layout()

    # ============================================================
    # Return stats
    stats = {
        "n_nodes": ego_graph.number_of_nodes(),
        "n_edges": ego_graph.number_of_edges(),
        "n_secondary": len(secondary),
        "n_primary": len(primary),
        "n_users": len(user_nodes),
        "secondary": secondary,
        "primary": primary,
    }

    return ax, stats

### plot

In [ ]:
org_keywords = [
    "youtube",
    "vox",
    "nyt",
    "cnn",
    "fox",
    "msnbc",
    "gop",
    "news",
    "times",
    "variety",
    "wsj",
    "usatoday",
    "womensmarch",
    "thedailybeast",
    "dnc",
    "foxnews",
    "ap",
    "metoomvmt",
    "cbs",
    "abc",
    "thr",
    "timesupnow",
    "washingtonpost",
]
ax, stats = make_network(
    G,
    df_top_mentioned,
    n_top_accounts=30,
    figure_size=24,
    spring_k=2,
    spring_iterations=100,
    random_seed=42,
    blue_node_size_mult=15,
    blue_node_alpha=0.45,
    red_node_size_mult=10,
    red_node_alpha=0.25,
    secondary_node_alpha=0.32,
    secondary_node_color="#332288",
    edge_alpha=0.06,
    label_font_size=23,
    label_font_color=".1",
    secondary_label_color=".1",
    use_adjust_text=True,
    adjust_text_expand_points=(0.3, 0.3),  # Push labels 1.5× away from nodes
    adjust_text_expand_text=(1.0, 1.0),  # Keep labels 1.3× apart from each other
    adjust_text_force=(0.1, 0.1),
    secondary_keywords=org_keywords,
)
save_mpl_fig("./output/figures/network_mentions")